# $C_3$ on Facebook

This notebook explores the **3rd-order higher-order clustering coefficient** $C_3$ on the Facebook ground-truth graph. This metric is the cleanest “beyond triangles” extension in the current bundle.

## Metric definition

Following Yin, Benson, and Leskovec (2018),

$$C_\ell(G)=\frac{(\ell^2+\ell)|K_{\ell+1}(G)|}{|W_\ell(G)|}. $$

For $\ell=3$,

$$C_3(G)=\frac{12|K_4(G)|}{|W_3(G)|}. $$

Interpretation: start from a triangle-centered higher-order wedge; how often does it close into a 4-clique?

In [ ]:

from pathlib import Path
import sys
import math
import itertools
import shutil

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'metrics.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent

sys.path.insert(0, str(NOTEBOOK_DIR))
from metrics import (
    load_graph,
    unique_triangle_count,
    wedge_count,
    gcc,
    alcc,
    count_k_cliques,
    k_clique_density,
    higher_order_global_clustering,
    higher_order_average_local_clustering,
    higher_order_local_clustering,
    node_k_clique_membership_counts,
    gcd11,
    orca_node_orbits_4,
    gcm11_from_orbits,
    GCD11_ORBITS,
)

DATA_PATH = NOTEBOOK_DIR.parent / 'data' / 'gt_txt' / 'facebook.txt'
G = load_graph(DATA_PATH)
print(f'Loaded {DATA_PATH.name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')


In [ ]:

def induced_ego_subgraph(G, center=None, radius=1, max_nodes=40):
    if center is None:
        center = max(G.degree, key=lambda x: x[1])[0]
    nodes = set(nx.single_source_shortest_path_length(G, center, cutoff=radius).keys())
    H = G.subgraph(nodes).copy()
    if H.number_of_nodes() > max_nodes:
        nbrs = sorted(H.nodes(), key=lambda u: (-G.degree[u], u))[:max_nodes]
        H = G.subgraph(nbrs).copy()
    return center, H


def edge_df(G):
    return pd.DataFrame(sorted((min(u,v), max(u,v)) for u,v in G.edges()), columns=['u','v'])


In [ ]:
c3_global = higher_order_global_clustering(G, 3)
c3_local_avg = higher_order_average_local_clustering(G, 3)
pd.DataFrame([
    {'metric': 'C3 global', 'value': c3_global},
    {'metric': 'C3 average local', 'value': c3_local_avg},
])

## Nodewise ingredients

For a node $u$, one convenient formula is

$$C_3(u)=\frac{|K_4(u)|}{(d_u-2)|K_3(u)|}. $$

So we need:
- how many triangles contain the node
- how many 4-cliques contain the node
- the node degree

In [ ]:
K3_u = node_k_clique_membership_counts(G, 3)
K4_u = node_k_clique_membership_counts(G, 4)
C3_u = higher_order_local_clustering(G, 3)
node_df = pd.DataFrame({
    'node': sorted(G.nodes()),
    'degree': [G.degree[u] for u in sorted(G.nodes())],
    'K3_memberships': [K3_u[u] for u in sorted(G.nodes())],
    'K4_memberships': [K4_u[u] for u in sorted(G.nodes())],
    'C3_local': [C3_u.get(u, np.nan) for u in sorted(G.nodes())],
}).sort_values('C3_local', ascending=False)
node_df.head(20)

In [ ]:
plt.figure(figsize=(7,4))
vals = node_df['C3_local'].dropna()
plt.hist(vals, bins=30)
plt.title('Distribution of local $C_3(u)$ values on Facebook')
plt.xlabel('$C_3(u)$')
plt.ylabel('Number of nodes')
plt.show()

## Tactile node example

Pick one node with a high local $C_3$ and verify the numerator and denominator manually.

In [ ]:
row = node_df.dropna().iloc[0]
u = int(row['node'])
du = G.degree[u]
num = K4_u[u]
denom = (du - 2) * K3_u[u]
manual = num / denom if denom > 0 else np.nan
pd.DataFrame([
    {'quantity': 'node', 'value': u},
    {'quantity': 'degree d_u', 'value': du},
    {'quantity': '|K3(u)|', 'value': K3_u[u]},
    {'quantity': '|K4(u)|', 'value': K4_u[u]},
    {'quantity': '(d_u - 2) * |K3(u)|', 'value': denom},
    {'quantity': 'manual C3(u)', 'value': manual},
    {'quantity': 'computed C3(u)', 'value': C3_u[u]},
])

## Interpretation

$C_3$ complements 4-clique density:
- 4-clique density asks how common 4-cliques are among all 4-node sets.
- $C_3$ asks how often existing triangle-based structure *expands* into 4-cliques.

For realism testing, matching Facebook's $C_3$ means reproducing its tendency for triangles to sit inside denser 4-node groups.